In [ ]:
import torch
import torch.nn as nn
import numpy as np
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
def format_matrix(M, name, row_prefix= 'rule', col_prefix= 'dim'):
    M = np.atleast_2d(M)
    n_rows, n_cols = M.shape

    header = '      ' + ' '.join(f'{col_prefix}{j:>2}' for j in range(n_cols))
    lines = [f'{name}:', header]
    for i, row in enumerate(M):
        row_str = ' '.join(f'{v:5.2f}' for v in row)
        lines.append(f'{row_prefix}{i:>2} | {row_str}')
    
    return '\n'.join(lines)


In [ ]:
def build_update_MLP(input_size, output_size= 2, hidden_size= 5):
    return nn.Sequential(
        # h1
        nn.Linear(input_size, hidden_size),
        nn.ReLU(),
        # h2
        nn.Linear(hidden_size, hidden_size),
        nn.ReLU(),
        # h3
        nn.Linear(hidden_size, hidden_size),
        nn.ReLU(),
        # y
        nn.Linear(hidden_size, output_size)
    )

def build_choice_MLP(input_size, output_size, hidden_size= 2):
    return nn.Sequential(
        # h1
        nn.Linear(input_size, hidden_size),
        nn.ReLU(),
        # h2
        nn.Linear(hidden_size, hidden_size),
        nn.ReLU(),
        # y
        nn.Linear(hidden_size, output_size)
    )

In [ ]:
class DisRNN(nn.Module):
    def __init__(self, m, n, q):
        super().__init__()

        self.m = m
        self.n = n
        self.q = q

        # build update MLPs and choice MLP
        self.updateMLPs = nn.ModuleList(
            [build_update_MLP(m+n) for _ in range(m)]
        )
        self.choiceMLP = build_choice_MLP(m,q)
            
        # initialize bottleneck parameters
        self.logit_M_h = nn.Parameter(5 * torch.ones((m,m)))  # M(i,j) -> update rule i's dependence on latent j
        self.log_sigma_h = nn.Parameter(-2 * torch.ones((m,m)))

        self.logit_M_x = nn.Parameter(5 * torch.ones((m,n)))
        self.log_sigma_x = nn.Parameter(-2 * torch.ones((m,n)))

        self.logit_M_z = nn.Parameter(5 *torch.ones(m))
        self.log_sigma_z = nn.Parameter(-2 * torch.ones(m))


    def bottleneck(self, x, m, sigma):
        if m.dim() == 1:
            noise = torch.randn_like(x) if self.training else torch.zeros_like(x)
            mean = m.unsqueeze(0) * x
            std = sigma.unsqueeze(0)
        else:
            shape = (m.shape[0], *x.shape)

            noise = torch.randn(shape, device= x.device) if self.training else torch.zeros(shape, device= x.device)
            mean = m.unsqueeze(1) * x.unsqueeze(0)
            std = sigma.unsqueeze(1)

        z = mean + std * noise
        kl = 0.5 * torch.sum(
            mean**2 + std**2 - torch.log(std**2 + 1e-8) - 1, 
            dim= -1
        )
        
        return z, kl

    def step(self, h, x):
        h_prev = h

        # bottlenecks for disentangled update rules
        M_h = torch.sigmoid(self.logit_M_h)
        sigma_h = torch.exp(self.log_sigma_h)
        h, kl_h = self.bottleneck(h, M_h, sigma_h)

        M_x = torch.sigmoid(self.logit_M_x)
        sigma_x = torch.exp(self.log_sigma_x)
        x, kl_x = self.bottleneck(x, M_x, sigma_x)

        # apply update MLPs
        z_outs = []
        for i, MLP in enumerate(self.updateMLPs):
            z_i = torch.cat([h[i], x[i]], dim= -1)
            logit_w, u = MLP(z_i).unbind(dim= -1)
            w = torch.sigmoid(logit_w)
            z_out = (1 - w) * h_prev[:, i] + w * u
            z_outs.append(z_out)
        z = torch.stack(z_outs, dim= -1)

        # bottlenecks for disentangled latents
        M_z = torch.sigmoid(self.logit_M_z)
        sigma_z = torch.exp(self.log_sigma_z)
        z, kl_z = self.bottleneck(z, M_z, sigma_z)

        kls = {
            'h': kl_h.sum(dim= 0),
            'x': kl_x.sum(dim= 0),
            'z': kl_z
        }

        return z, kls


    def out(self, h):
        return self.choiceMLP(h)


    def forward(self, X):
        T, B, n = X.shape
        assert n == self.n

        latents = []
        bottleneck_losses = {'h': [], 'x': [], 'z': []}
        outputs = []
        
        h = torch.zeros(B, self.m, device= X.device)
        latents.append(h)
        for t in range(T):
            x = X[t]

            h, kls = self.step(h, x)
            latents.append(h)
            for key, val in kls.items():
                bottleneck_losses[key].append(val)

            y = self.out(h)
            outputs.append(y)

        bottleneck_losses = {key: torch.stack(vals) for key, vals in bottleneck_losses.items()}

        return torch.stack(outputs), torch.stack(latents), bottleneck_losses


In [ ]:
# set task space
D = torch.rand
num_arms = 2

# initialize DisRNN model and Adam optimizer
hidden_size = 5
input_size = 2
model = DisRNN(m= hidden_size, n= 2, q= num_arms)
optimizer = torch.optim.Adam(model.parameters(), lr= 5e-3)

# setup GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


In [ ]:
# training hyperparameters
episodes = 2000
batch_size = 1024
batch_idx = torch.arange(batch_size, device= device)
T = 500
drift = 0.02
beta_max = 1e-3
beta_warmup_start = 200
beta_warmup_end = 500

for ep in range(episodes):
    model.train()
    optimizer.zero_grad()

    probs = D(batch_size, num_arms, device= device)

    h = torch.zeros(batch_size, hidden_size, device= device)
    x = torch.zeros(batch_size, input_size, device= device)

    log_probs = []
    rewards = []
    bottleneck_losses = {'h': [], 'x': [], 'z': []}

    for t in range(T):
        if t % 50 == 0:
            h = h.detach()

        h, kls = model.step(h, x)
        logits = model.out(h)

        # sample action and get rewards
        pi = torch.distributions.Categorical(logits= logits)
        a = pi.sample()
        log_prob = pi.log_prob(a)
        r = (torch.rand(batch_size, device= device) < probs[batch_idx, a]).float()

        # prepare next input
        x = torch.stack([2*a.float() - 1, 2*r - 1], dim= -1)

        # apply bounded random drift to probs
        probs += drift * torch.randn(batch_size, num_arms, device= device)
        probs = torch.clamp(probs, 0, 1)

        log_probs.append(log_prob)
        rewards.append(r)
        for key, val in kls.items():
            bottleneck_losses[key].append(val)

    log_probs = torch.stack(log_probs)
    rewards = torch.stack(rewards)
    bottleneck_losses = {key: torch.stack(vals) for key, vals in bottleneck_losses.items()}

    # REINFORCE with baseline
    baseline = rewards.mean(dim= 0, keepdim= True)
    advantage = rewards - baseline

    # update beta
    beta = beta_max * min((ep - beta_warmup_start) / (beta_warmup_end - beta_warmup_start), 1.0)
    beta = 0 if ep < beta_warmup_start else beta

    loss_rl = -(log_probs * advantage).mean()
    loss_bottleneck = sum(loss.mean() for loss in bottleneck_losses.values())
    loss = loss_rl + beta * loss_bottleneck

    loss.backward()
    optimizer.step()

    if ep % 50 == 0:
        mean_reward = rewards.mean().item()

        M_h = torch.sigmoid(model.logit_M_h).detach().cpu().numpy()
        M_x = torch.sigmoid(model.logit_M_x).detach().cpu().numpy()
        M_z = torch.sigmoid(model.logit_M_z).detach().cpu().numpy()

        print(
            f'ep {ep:5d} | loss {loss.item():.4f} | '
            f'RL {loss_rl.item():.4f} | KL {loss_bottleneck.item():.4f} | '
            f'beta {beta:.2e} | mean_r {mean_reward:.3f}'
        )
        print(format_matrix(M_h, 'M_h', row_prefix='rule', col_prefix='lat'))
        print(format_matrix(M_x, 'M_x', row_prefix='rule', col_prefix='obs'))
        print(format_matrix(M_z.reshape(1,-1), 'M_z', row_prefix='lat', col_prefix='lat'))
        print()

    if (ep % 250 == 0 and ep > 0) or ep == episodes - 1:
        torch.save({
            'ep': ep,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss
        }, f'/content/drive/MyDrive/checkpoint_ep{ep}.pt')


In [ ]:
# reinitialize
D = torch.rand
num_arms = 2

hidden_size = 5
input_size = 2
model = DisRNN(m= hidden_size, n= 2, q= num_arms)
optimizer = torch.optim.Adam(model.parameters(), lr= 5e-3)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

episodes = 3000
batch_size = 1024
batch_idx = torch.arange(batch_size, device= device)
T = 500
drift = 0.02
beta_max = 1e-3
beta_warmup_start = 200
beta_warmup_end = 500


# load checkpoint and resume training
episode = 1999
checkpoint = torch.load(f'/content/drive/MyDrive/checkpoint_ep{episode}.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

for ep in range(episode + 1, episodes):
    model.train()
    optimizer.zero_grad()

    probs = D(batch_size, num_arms, device= device)

    h = torch.zeros(batch_size, hidden_size, device= device)
    x = torch.zeros(batch_size, input_size, device= device)

    log_probs = []
    rewards = []
    bottleneck_losses = {'h': [], 'x': [], 'z': []}

    for t in range(T):
        if t % 50 == 0:
            h = h.detach()

        h, kls = model.step(h, x)
        logits = model.out(h)

        # sample action and get rewards
        pi = torch.distributions.Categorical(logits= logits)
        a = pi.sample()
        log_prob = pi.log_prob(a)
        r = (torch.rand(batch_size, device= device) < probs[batch_idx, a]).float()

        # prepare next input
        x = torch.stack([2*a.float() - 1, 2*r - 1], dim= -1)

        # apply bounded random drift to probs
        probs += drift * torch.randn(batch_size, num_arms, device= device)
        probs = torch.clamp(probs, 0, 1)

        log_probs.append(log_prob)
        rewards.append(r)
        for key, val in kls.items():
            bottleneck_losses[key].append(val)

    log_probs = torch.stack(log_probs)
    rewards = torch.stack(rewards)
    bottleneck_losses = {key: torch.stack(vals) for key, vals in bottleneck_losses.items()}

    # REINFORCE with baseline
    baseline = rewards.mean(dim= 0, keepdim= True)
    advantage = rewards - baseline

    # update beta
    beta = beta_max * min((ep - beta_warmup_start) / (beta_warmup_end - beta_warmup_start), 1.0)
    beta = 0 if ep < beta_warmup_start else beta

    loss_rl = -(log_probs * advantage).mean()
    loss_bottleneck = sum(loss.mean() for loss in bottleneck_losses.values())
    loss = loss_rl + beta * loss_bottleneck

    loss.backward()
    optimizer.step()

    if ep % 50 == 0:
        mean_reward = rewards.mean().item()

        M_h = torch.sigmoid(model.logit_M_h).detach().cpu().numpy()
        M_x = torch.sigmoid(model.logit_M_x).detach().cpu().numpy()
        M_z = torch.sigmoid(model.logit_M_z).detach().cpu().numpy()

        print(
            f'ep {ep:5d} | loss {loss.item():.4f} | '
            f'RL {loss_rl.item():.4f} | KL {loss_bottleneck.item():.4f} | '
            f'beta {beta:.2e} | mean_r {mean_reward:.3f}'
        )
        print(format_matrix(M_h, 'M_h', row_prefix='rule', col_prefix='lat'))
        print(format_matrix(M_x, 'M_x', row_prefix='rule', col_prefix='obs'))
        print(format_matrix(M_z.reshape(1,-1), 'M_z', row_prefix='lat', col_prefix='lat'))
        print()

    if (ep % 250 == 0 and ep > 0) or ep == episodes - 1:
        torch.save({
            'ep': ep,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss
        }, f'/content/drive/MyDrive/checkpoint_ep{ep}.pt')
